# POLITE — Lab Bench Control

Interactive control of the lab setup (no telescope): **QHY268M** camera + **ZWO EFW**
filter wheel over the INDIGO Alpaca agent, plus the **Optec Pyxis 2" Gen3** half-wave
plate over its native serial link.

**Kernel:** select the `POLITE` conda environment.

**Before you start**
- INDIGO is running with `indigo_agent_alpaca` exposing the camera + wheel
  (`http://localhost:11111`). Device numbers live in `obs_utils/user_config.py`.
- The Pyxis USB cable is plugged in and powered (`ls /dev/cu.usbserial*`).
- **One kernel owns the hardware** — don't run another control notebook or script
  against the same devices at once. *Restart the kernel* to fully reset.

Each action below is its own re-runnable cell: set slot 2 in one cell, slot 3 in
the next, with **no reconnect**.

In [ ]:
# --- POLITE path bootstrap ---
# Run from the repo root so `from obs_utils import ...` resolves. Jupyter starts
# the kernel in this notebook's own directory, so hop up one level when needed.
import os, sys
from pathlib import Path
_cwd = Path.cwd()
if _cwd.name == "notebooks":
    os.chdir(_cwd.parent)
_root = str(Path.cwd())
if _root not in sys.path:
    sys.path.insert(0, _root)
print("POLITE root:", _root)

## 1 · Connect  *(run once)*

In [ ]:
from obs_utils import interactive as obs

# Alpaca instrument + native-serial Pyxis HWP. Set USE_HWP=False to skip the
# rotator (e.g. HWP not plugged in). Re-running this cell is safe: live
# connections are reused, so the exclusive serial port is not double-opened.
USE_HWP = True
s = obs.connect(alpaca=True, pyxis_serial=USE_HWP)

## 1b · Per-component connect  *(fault-isolated fallback)*

`connect()` above brings up the **whole** Alpaca instrument at once — if the
camera is offline you get nothing, even when the wheel is fine. When one device
misbehaves, skip section 1 and use the cells below instead: each connects **one**
component and attaches it to the same shared session `s`. Run only the ones whose
hardware is up; a failure in one cell is isolated to that cell, and the devices
that did connect keep working (`s.status()`, `s.filter`, `s.hwp`, `s.expose` all
operate on whatever is live). Each cell is safe to re-run.

In [ ]:
s = obs.connect_camera()          # QHY268M only

In [ ]:
s = obs.connect_filter_wheel()    # ZWO EFW only

In [ ]:
s = obs.connect_hwp_serial()      # native-serial Pyxis Gen3 HWP only

## 2 · Status

In [ ]:
s.status()

## 3 · Filter wheel — change slots live

Slots are 0-based; names come from `user_config.filter_names`. Pass an index or a name.

In [ ]:
s.filter(2)                 # -> Photometric R

In [ ]:
s.filter(3)                 # run AFTER the cell above — no reconnect

In [ ]:
s.filter("Photometric V")   # ...or by name

In [ ]:
s.current_filter()

## 4 · Half-wave plate (Pyxis Gen3)

The Gen3 rejects moves until it is **homed** — home once per power cycle. Both
cells cause **physical motion**.

In [ ]:
s.home_hwp()      # physical motion; seconds to ~a minute

In [ ]:
s.hwp(90.0)       # absolute instrument PA [deg]

## 5 · Camera — take a frame

In [ ]:
img = s.expose(1.0, dark=True)     # ndarray; dark = shutter closed
img.shape, img.dtype

In [ ]:
# ...or write FITS straight to disk:
s.expose(1.0, out_path="lab_trial.fits", dark=True)

## 6 · Shutdown

Releases the camera, wheel, and serial port. Safe to leave the kernel running;
restart the kernel for a fully clean slate.

In [ ]:
obs.shutdown()